Implement batch gradient descent with early stopping for softmax regression without using Scikit-Learn, only NumPy. Use it on a classification task such as the iris dataset.

In [213]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import add_dummy_feature
from sklearn.model_selection import train_test_split

In [214]:
iris_data = load_iris()

In [215]:
iris_data.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [216]:
iris_data.feature_names

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [217]:
X, y = iris_data.data, iris_data.target

In [220]:
x_train, x_val, y_train, y_val = train_test_split(X_b, y , test_size=0.2, random_state=42)

In [221]:
n_epochs = 100
n_samples = X.shape[0]
n_features = X.shape[1]
batch_size = 15
n_classes = 3

In [222]:
def softmax(z):
    z = z - np.max(z, axis=-1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

In [251]:

def softmax_regression(
    x_train,
    y_train,
    x_val,
    y_val,
    alpha: float = 0.01,
    n_epochs: int = 1000,
    patience: int = 10,
    min_delta: float = 1e-4,
):
    rng = np.random.default_rng(seed=42)

    n_samples = len(x_train)
    n_classes = len(np.unique(y_train))
    n_features = x_train.shape[1] + 1

    thetas = rng.standard_normal((n_classes, n_features))

    X_b = add_dummy_feature(x_train)
    X_val_b = add_dummy_feature(x_val)

    best_loss = np.inf
    best_thetas = None
    epochs_without_improvement = 0

    for epoch in range(n_epochs):

        # Batch Gradient Descent
        z = X_b @ thetas.T
        p = softmax(z)

        Y = np.eye(n_classes)[y_train]
        grad = (p - Y).T @ X_b / n_samples

        thetas -= alpha * grad

        # Validation Loss
        z_val = X_val_b @ thetas.T
        p_val = softmax(z_val)

        val_loss = -np.mean(
            np.log(
                p_val[np.arange(len(y_val)), y_val] + 1e-15
            )
        )

        # Early Stopping
        if val_loss < best_loss - min_delta:
            best_loss = val_loss
            best_thetas = thetas.copy()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch + 1}: "
            f"Validation Loss = {val_loss:.5f}"
        )

        if epochs_without_improvement >= patience:
            print("Early stopping triggered!")
            break

    return best_thetas


In [252]:
best_thetas = softmax_regression(x_train, y_train, x_val, y_val)

Epoch 1: Validation Loss = 0.93934
Epoch 2: Validation Loss = 0.90474
Epoch 3: Validation Loss = 0.87956
Epoch 4: Validation Loss = 0.86107
Epoch 5: Validation Loss = 0.84726
Epoch 6: Validation Loss = 0.83670
Epoch 7: Validation Loss = 0.82842
Epoch 8: Validation Loss = 0.82174
Epoch 9: Validation Loss = 0.81621
Epoch 10: Validation Loss = 0.81152
Epoch 11: Validation Loss = 0.80745
Epoch 12: Validation Loss = 0.80385
Epoch 13: Validation Loss = 0.80061
Epoch 14: Validation Loss = 0.79765
Epoch 15: Validation Loss = 0.79491
Epoch 16: Validation Loss = 0.79236
Epoch 17: Validation Loss = 0.78995
Epoch 18: Validation Loss = 0.78766
Epoch 19: Validation Loss = 0.78549
Epoch 20: Validation Loss = 0.78340
Epoch 21: Validation Loss = 0.78138
Epoch 22: Validation Loss = 0.77944
Epoch 23: Validation Loss = 0.77756
Epoch 24: Validation Loss = 0.77573
Epoch 25: Validation Loss = 0.77395
Epoch 26: Validation Loss = 0.77222
Epoch 27: Validation Loss = 0.77053
Epoch 28: Validation Loss = 0.76887
E

In [262]:
x = add_dummy_feature(x_val[2].reshape(1, -1))

z = x @ best_thetas.T
p = softmax(z)

predicted_class = np.argmax(p)

print("Probabilities:", p)
print("Predicted class:", iris_data.target_names[predicted_class])
print("Actual class:", iris_data.target_names[y_val[2]])

Probabilities: [[6.07507934e-06 1.52154860e-01 8.47839065e-01]]
Predicted class: virginica
Actual class: virginica
